# Banking loan default: ANN with five hidden layers

**Goal:** predict `defaulted` (`1` = default, `0` = no default) from four applicant features. This is a **synthetic teaching dataset**, not real customer data. All four notebooks use the same CSV, feature order, random seed, train/test split, and scaling so model structures can be compared fairly.

Keep `Banking_Loan_Default_2400.csv` in the same folder as this notebook. Run the cells in order with Windows with Python 3.13.5. The final metric values depend on installed library versions and stochastic training.

## Windows setup (Python 3.13.5)
In Windows Command Prompt or PowerShell, run:
```powershell
py -3.13 -m pip install --upgrade pip
py -3.13 -m pip install notebook pandas numpy scikit-learn matplotlib tensorflow
```
Start Jupyter with `py -3.13 -m notebook`. Select the Python 3.13 kernel. TensorFlow includes the `tf.keras` API used in the ANN notebooks. A regular Windows CPU is sufficient for this small exercise.

## Step 1 - Import libraries
`pandas` reads the CSV; NumPy handles arrays; scikit-learn splits, scales, and evaluates. The ANN notebooks additionally use `tf.keras`, the Keras API included with TensorFlow.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt

## Step 2 - Read and inspect the banking CSV
Each row represents one fictional applicant. `defaulted` is the answer to predict, so it must never be included among the input features.

In [ ]:
df = pd.read_csv("Banking_Loan_Default_2400.csv")
print("Rows and columns:", df.shape)
display(df.head())
print("Missing values:\n", df.isna().sum())
print("Default counts:\n", df["defaulted"].value_counts())

## Step 3 - Choose inputs and split the data
`X` contains exactly four input features; `y` contains the target. `test_size=0.20` reserves 20% for final evaluation. `stratify=y` keeps approximately the same default rate in train and test. `random_state=42` makes the split repeatable.

In [ ]:
features = ["age_years", "annual_income_usd", "outstanding_debt_usd", "credit_score"]
X = df[features]
y = df["defaulted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train), "Test rows:", len(X_test))

## Step 4 - Standardize inputs without leaking test information
A credit score and income have very different numeric ranges. `StandardScaler` puts them on comparable scales. Fit it on training data only, then use that fitted scaler on the test data. A standardized value means the number of training standard deviations above or below the training mean.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(pd.DataFrame(X_train_scaled[:3], columns=features).round(2))

## Step 5 - Import TensorFlow and set the random seed
`tf.keras` is provided by TensorFlow. The seed helps make runs more repeatable; minor differences can remain across devices or library versions.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(42)
print("TensorFlow version:", tf.__version__)

## Step 6 - Build the 5-hidden-layer network
Architecture: **4 inputs → 16, 12, 10, 8, 6 hidden neurons → 1 sigmoid output**. Each `Dense` layer connects to every value from the preceding layer. Each hidden neuron computes `z = sum(input × weight) + bias`, then applies ReLU: `max(0, z)`. The output sigmoid gives a probability. `Input` describes the feature shape and is not a hidden layer.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(len(features),)),
    layers.Dense(16, activation="relu", name="hidden_1"),
    layers.Dense(12, activation="relu", name="hidden_2"),
    layers.Dense(10, activation="relu", name="hidden_3"),
    layers.Dense(8, activation="relu", name="hidden_4"),
    layers.Dense(6, activation="relu", name="hidden_5"),
    layers.Dense(1, activation="sigmoid", name="default_probability"),
])
model.summary()

## Step 7 - Compile the model
`binary_crossentropy` compares predicted probabilities with 0/1 labels. Adam updates the learned weights and biases to reduce this loss. `learning_rate=0.001` controls update size. Accuracy is tracked for easier reading.

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss="binary_crossentropy", metrics=["accuracy"])

## Step 8 - Train with a validation split
`epochs=35` means up to 35 passes over the training subset. `batch_size=32` updates weights after batches of 32 examples. `validation_split=0.20` holds out 20% of the training rows to watch generalization; the separate test set remains untouched. Early stopping restores the best validation-loss weights if progress stalls.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
history = model.fit(X_train_scaled, y_train.to_numpy(),
                    epochs=35, batch_size=32, validation_split=0.20,
                    callbacks=[early_stop], verbose=1)

## Step 9 - Plot learning curves
Training loss measures fit on the training subset; validation loss measures fit on held-out training rows. If training loss falls while validation loss rises, the network may be overfitting.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.title("Learning curves")
plt.legend()
plt.tight_layout()
plt.show()

## Step 10 - Evaluate on the untouched test set
The output neuron predicts a probability for each applicant. The 0.5 cutoff converts probabilities into classes for the confusion matrix and classification report. Compare ROC AUC and default-class recall as well as accuracy.

In [ ]:
probabilities = model.predict(X_test_scaled, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)
print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("ROC AUC:", round(roc_auc_score(y_test, probabilities), 3))
print("Confusion matrix:\n", confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions, target_names=["No default", "Default"]))

## Step 11 - Open one hidden neuron: inputs, weights, bias, activation
`get_weights()` returns a weight matrix and bias vector for the first hidden layer. Column 0 is the first hidden neuron. Its calculation is `z = sum(scaled inputs × neuron weights) + neuron bias`; for ReLU, `activation = max(0, z)`. The following cell checks this manual calculation against the layer output.

In [ ]:
first_layer = model.get_layer("hidden_1")
weight_matrix, bias_vector = first_layer.get_weights()
example = X_test_scaled[0].astype("float32")
first_neuron_weights = weight_matrix[:, 0]
z = np.dot(example, first_neuron_weights) + bias_vector[0]
a = max(0, z)
display(pd.DataFrame({"feature": features, "scaled_input": example, "weight_to_neuron_1": first_neuron_weights, "contribution": example * first_neuron_weights}).round(4))
print("First hidden neuron bias:", round(float(bias_vector[0]), 4))
print("Weighted sum z:", round(float(z), 4))
print("ReLU activation a:", round(float(a), 4))
probe = keras.Model(inputs=model.inputs, outputs=first_layer.output)
print("Keras first-neuron activation:", round(float(probe.predict(example.reshape(1, -1), verbose=0)[0, 0]), 4))

## Step 12 - Inspect layer shapes and parameter counts
For a Dense layer: **parameters = input values × neurons + one bias per neuron**. The output layer also has its own weights and bias. Count only `hidden_...` layers when stating network depth.

In [ ]:
for layer in model.layers:
    weights, biases = layer.get_weights()
    print(f"{layer.name}: weights {weights.shape}, biases {biases.shape}, parameters {layer.count_params()}")

## Step 13 - What changed?
This network has **5 hidden layers**, compared with zero in logistic regression. Extra layers make the model deeper and can combine intermediate patterns. The same test data should be used for comparison. A deeper model is not automatically more accurate or more suitable for credit decisions; this synthetic exercise is for learning architecture and calculations.